In [2]:
import h5py
import numpy as np
import pandas as pd
import sys
import os
import time
import requests
import urllib.parse
import zipfile

In [23]:
API_KEY = 'i5UTiFkCtT8I3BNCxhVZBOn2LbnSOk0EglUQyuEV'
EMAIL = "mukiibirogerz@gmail.com"

In [43]:
BASE_URL = "https://developer.nrel.gov/api/nsrdb/v2/solar/nsrdb-msg-v1-0-0-download.json?"


def download_data(years=[2019], location= ['1788603']):
    input_data = {
        'attributes': 'solar_zenith_angle,surface_albedo,total_precipitable_water,clearsky_dhi,clearsky_dni,clearsky_ghi,cloud_type,dew_point,relative_humidity,surface_pressure,dhi,dni,fill_flag,ghi,air_temperature,wind_direction,wind_speed',
        'interval': '60',

        'api_key': API_KEY,
        'email': EMAIL,
    }
    files = {}
    for name in years:
        print(f"Processing name: {name}")
        for id, location_ids in enumerate(location):
            input_data['names'] = [name]
            input_data['location_ids'] = location_ids
            print(f'Making request for point group {id + 1} of {len(location)}...')

            if '.csv' in BASE_URL:
                url = BASE_URL + urllib.parse.urlencode(data, True)
                # Note: CSV format is only supported for single point requests
                # Suggest that you might append to a larger data frame
                data = pd.read_csv(url)
                print(f'Response data (you should replace this print statement with your processing): {data}')
                # You can use the following code to write it to a file
                # data.to_csv('SingleBigDataPoint.csv')
            else:
                headers = {
                  'x-api-key': API_KEY
                }
                data = get_response_json_and_handle_errors(requests.post(BASE_URL, input_data, headers=headers))
                download_url = data['outputs']['downloadUrl']
                # You can do with what you will the download url
                print(data['outputs']['message'])
                print(f"Data can be downloaded from this url when ready: {download_url}")

                # Delay for 1 second to prevent rate limiting
                time.sleep(1)
                 # Download the file
                files[(name, location_ids)] = download_file(download_url, f"{name}_{location_ids}.zip")
            print(f'Processed')

    return files



def get_response_json_and_handle_errors(response: requests.Response) -> dict:
    """Takes the given response and handles any errors, along with providing
    the resulting json

    Parameters
    ----------
    response : requests.Response
        The response object

    Returns
    -------
    dict
        The resulting json
    """
    if response.status_code != 200:
        print(f"An error has occurred with the server or the request. The request response code/status: {response.status_code} {response.reason}")
        print(f"The response body: {response.text}")
        exit(1)

    try:
        response_json = response.json()
    except:
        print(f"The response couldn't be parsed as JSON, likely an issue with the server, here is the text: {response.text}")
        exit(1)

    if len(response_json['errors']) > 0:
        errors = '\n'.join(response_json['errors'])
        print(f"The request errored out, here are the errors: {errors}")
        exit(1)
    return response_json

def download_file(url: str, local_filename: str):
    """Downloads the file from the given URL and saves it locally

    Parameters
    ----------
    url : str
        The URL to download the file from
    local_filename : str
        The local file name to save the file as
    """
    # Add a retry mechanism with a maximum of 3 attempts
    max_attempts = 7
    for attempt in range(max_attempts):
        try:
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                with open(local_filename, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            return f
        except requests.exceptions.HTTPError as e:
            if attempt == max_attempts - 1:
                raise e
            else:
                print('Error downloading file, retrying...')
                time.sleep(2 ** attempt)


In [44]:
location = '1788603'
years = [2021]
data = download_data(years=years)

Processing name: 2021
Making request for point group 1 of 1...
File generation in progress. An email will be sent to mukiibirogerz@gmail.com when the download is ready.
Data can be downloaded from this url when ready: https://mapfiles.nrel.gov/data/solar/7b9b8a6932d1384c7375d38dfbbcc2d3.zip
Error downloading file, retrying...
Error downloading file, retrying...
Error downloading file, retrying...
Error downloading file, retrying...
Error downloading file, retrying...
Processed


In [45]:
def process_data(year, location):
    zip_filename = f"{year}_{location}.zip"
    extract_path = f"{year}_{location}"
    with zipfile.ZipFile(zip_filename, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    csv_filename = f'{extract_path}/{os.listdir(extract_path)[0]}/{location}_0.33_32.58_{year}.csv'
    data = pd.read_csv(csv_filename, skiprows=2)
    data['Date Local'] = pd.to_datetime(data[['Year', 'Month', 'Day', 'Hour', 'Minute']].assign(Day=lambda x: x.Day.astype(str).str.zfill(2)))
    data.drop(columns=['Year', 'Month', 'Day', 'Hour', 'Minute'], inplace=True)
    data.set_index('Date Local', inplace=True)
    return data

In [46]:
data = process_data(years[0], location)

In [47]:
data

,Solar Zenith Angle,Surface Albedo,Precipitable Water,Clearsky DHI,Clearsky DNI,Clearsky GHI,Cloud Type,Dew Point,Relative Humidity,Pressure,DHI,DNI,Fill Flag,GHI,Temperature,Wind Direction,Wind Speed
Date Local,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00,141.74,0.15,4.0,0,0,0,6,18.4,98.71,879,0,0,0,0,18.6,351,1.5
2021-01-01 00:30:00,135.68,0.15,3.9,0,0,0,6,18.4,99.33,879,0,0,0,0,18.5,349,1.4
2021-01-01 01:00:00,129.30,0.15,3.9,0,0,0,6,18.2,98.92,879,0,0,0,0,18.4,347,1.4
2021-01-01 01:30:00,122.72,0.15,3.9,0,0,0,7,18.2,98.92,879,0,0,0,0,18.4,352,1.3
2021-01-01 02:00:00,116.01,0.15,3.9,0,0,0,7,18.1,98.86,879,0,0,0,0,18.3,357,1.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-31 21:30:00,156.63,0.16,3.3,0,0,0,4,17.3,83.68,880,0,0,0,0,20.1,267,1.4
2021-12-31 22:00:00,157.24,0.16,3.3,0,0,0,3,17.2,84.37,880,0,0,0,0,19.9,270,1.4
2021-12-31 22:30:00,155.59,0.16,3.3,0,0,0,3,17.2,85.42,880,0,0,0,0,19.7,275,1.3
